# Step 10 — Tools

🇬🇧 **English** (this notebook)

Give an agent a tool to call at runtime — live web search, in this case. The agent decides *when* to call the tool and *what query* to use; this is what turns a frozen-knowledge model into one that can ground its answer in current information. Standalone: the tool is attached directly to the `Agent`, no `Crew` needed.

## Learning objective

By the end of this notebook, you will:

- Understand what a "tool" is in CrewAI, and why an agent — not you — decides when to call it
- Have attached a built-in tool (`SerperDevTool`, live web search) to an `Agent`
- Be able to tell, from the verbose log, exactly when and why the agent chose to search

## Prerequisites

- [Step 09 — Single Agent](step_09_single_agent.ipynb) completed — this notebook reuses its Researcher agent unchanged
- A `SERPER_API_KEY` in your `.env` — a free key at [serper.dev](https://serper.dev)
- The same `.env` setup as the previous steps otherwise

## Background

An agent's own knowledge is frozen at whatever point its training data ends — it cannot know about anything that happened afterward, and it has no way to check a fact against the current state of the world. A **tool** fixes that: a function the agent can choose to call mid-reasoning, whose result gets fed back into its next reasoning step.

> Schick, T., et al. (2023). *Toolformer: Language Models Can Teach Themselves to Use Tools*. [arXiv:2302.04761](https://arxiv.org/abs/2302.04761)

## How this works

`crewai_tools` ships `SerperDevTool` for web search out of the box — pass it directly to `tools=[...]` on the `Agent`:

1. The agent's identity (`role`/`goal`/`backstory`) is exactly Step 09's Researcher, unchanged.
2. The only addition is `tools=[SerperDevTool()]` — this is what makes web search available to the agent at all.
3. The `question` specifically asks about *current* deadlines — something a frozen training corpus can't fully know, so the agent has a real reason to reach for the tool rather than just answer from memory.
4. Nothing tells the agent *when* to search — that decision happens inside its own reasoning loop, visible in the verbose log once you run the cell.

In [1]:
import os

from dotenv import load_dotenv
from crewai import Agent
from crewai_tools import SerperDevTool

load_dotenv()

# ── Same "researcher" template as agents.yaml, {topic} filled in via an f-string ─
topic = "EU AI Act"

role      = f"{topic} Senior Data Researcher"
goal      = f"Uncover cutting-edge developments in {topic}"
backstory = (
    f"You're a seasoned researcher with a knack for uncovering the latest "
    f"developments in {topic}. Known for your ability to find the most relevant "
    f"information and present it in a clear and concise manner."
)

agent = Agent(
    role=role,
    goal=goal,
    backstory=backstory,
    tools=[SerperDevTool()],
    verbose=True,
)

# ── A question that needs live information, not training-data knowledge ──────
question = (
    f"Search for the most recent confirmed enforcement deadlines for {topic}. "
    "Which obligations have already come into force, and which are still upcoming?"
)

result = await agent.kickoff(question)
print(result.raw)

╭───────────────────────────────────────────── 🤖 LiteAgent Started ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Started                                                                                              │
│  Role: EU AI Act Senior Data Researcher                                                                         │
│  Status: In Progress                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 🤖 LiteAgent Started ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Session Started                                                                                      │
│  Name:                                                                                                          │
│  EU AI Act Senior Data Researcher                                                                               │
│  id:                                                                                                            │
│  4cfae625-454e-4f10-9a0f-90894b70eb3d                                                                           │
│  role:                                                                                                          │
│  EU AI Act Senior Data Researcher                                                                               │
│  goal:                                                                                                          │
│  Uncover cutting-edge developments in EU AI Act                                                                 │
│  backstory:                                                                                                     │
│  You're a seasoned researcher with a knack for uncovering the latest developments in EU AI Act. Known for your  │
│  ability to find the most relevant information and present it in a clear and concise manner.                    │
│  tools:                                                                                                         │
│  [SerperDevTool(name='Search the internet with Serper', description='Tool Name:                                 │
│  search_the_internet_with_serper\nTool Arguments: {\n  "description": "Input for SerperDevTool.",\n             │
│  "properties": {\n    "search_query": {\n      "description": "Mandatory search query you want to use to        │
│  search the internet",\n      "title": "Search Query",\n      "type": "string"\n    }\n  },\n  "required": [\n  │
│  "search_query"\n  ],\n  "title": "SerperDevToolSchema",\n  "type": "object",\n  "additionalProperties":        │
│  false\n}\nTool Description: A tool that can be used to search the internet with a search_query. Supports       │
│  different search types: \'search\' (default), \'news\'', env_vars=[EnvVar(name='SERPER_API_KEY',               │
│  description='API key for Serper', required=True, default=None)], args_schema=<class                            │
│  'crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevToolSchema'>, description_updated=False,          │
│  cache_function=<function BaseTool.<lambda> at 0x1132df240>, result_as_answer=False, max_usage_count=None,      │
│  current_usage_count=0, base_url='https://google.serper.dev', n_results=10, save_file=False,                    │
│  search_type='search', country='', location='', locale='')]                                                     │
│  verbose:                                                                                                       │
│  True                                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'EU AI Act enforcement timeline deadlines 2024 2025 2026'}                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'EU AI Act enforcement timeline deadlines 2024 2025 2026', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'EU Artificial Intelligence Act | Up-to-da...


╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Output: {'searchParameters': {'q': 'EU AI Act enforcement timeline deadlines 2024 2025 2026', 'type':          │
│  'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'EU Artificial Intelligence Act | Up-to-date   │
│  developments and ...', 'link': 'https://artificialintelligenceact.eu/', 'snippet': 'each Member State must     │
│  establish at least one AI regulatory sandbox at the national level by 2 August 2026. Application deadline is   │
│  13 December 2024.', 'position': 1}, {'title': "AI Act | Shaping Europe's digital future - European Union",     │
│  'link': 'https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai', 'snippet': 'The AI Act    │
│  entered into force on 1 August 2024, and will be fully applicable 2 years later on 2 August 2026, with some    │
│  exceptions: prohibited AI practices', 'position': 2}, {'title': 'Implementation Timeline | EU Artificial       │
│  Intelligence Act', 'link': 'https://artificialintelligenceact.eu/implementation-timeline/', 'snippet':         │
│  'Timeline of key dates. Last updated: 1 August 2024. Items in blue relate to the Application of the Act.       │
│  Download timeline as image. 2024. Date. 12 July 2024.', 'position': 3}, {'title': 'The roadmap to the EU AI    │
│  Act: a detailed guide', 'link': 'https://www.alexanderthamm.com/en/blog/eu-ai-act-timeline/', 'snippet':       │
│  'Timeline for the adoption of the European AI Act ; 13 March 2024, EU Parliament approves the draft law ; 12   │
│  July 2024, Publication of the law in ...', 'position': 4}, {'title': 'EU AI Act 2026 Updates: Compliance       │
│  Requirements and Business Risks', 'link':                                                                      │
│  'https://www.legalnodes.com/article/eu-ai-act-2026-updates-compliance-requirements-and-business-risks',        │
│  'snippet': 'AI Act application timeline and key deadlines · from 2 February 2025: · from 2 August 2025: · The  │
│  AI Act applies in full to all remaining ...', 'position': 5}, {'title': 'EU AI Act Timeline: Key Compliance    │
│  Dates & Deadlines Explained', 'link': 'https://www.dataguard.com/eu-ai-act/timeline', 'snippet': 'The EU AI    │
│  Act has a staggered implementation timeline. Some companies have until 2027 to adapt to its requirements.      │
│  Others have until 2026, while new AI ...', 'position': 6}, {'title': "The EU AI Act's August 2 Deadline",      │
│  'link': 'https://www.linkedin.com/pulse/eu-ai-acts-august-2-deadline-kai-hackbarth-oytuf', 'snippet': 'It      │
│  entered into force on August 1, 2024, Before August 2, 2026 preceded the August 2 date: February 2, 2025 —     │
│  Prohibited AI and AI Literacy.', 'position': 7}, {'title': 'Timeline for the Implementation of the EU AI Act   │
│  | AI Act Service Desk', 'link':                                                                                │
│  'https://ai-act-service-desk.ec.europa.eu/en/ai-act/timeline/timeline-implementation-eu-ai-act', 'snippet':    │
│  'Timeline for the Implementation of the EU AI Act ; 01 Aug 2024. Entry into force ; 02 Feb 2025. General       │
│  provisions (definitions & AI literacy) and prohibitions ...', 'position': 8}, {'title': 'EU AI Act Compliance  │
│  Timeline: Key Dates and How to Prepare', 'link': 'https://addcomply.com/eu-ai-act-compliance-timeline/',       │
│  'snippet': 'The EU AI Act entered into force in August 2024 and will apply progressively, with requirements    │
│  rolling out in phases — step by step.', 'position': 9}

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  As an EU AI Act researcher, I have summarized the enforcement timeline for you.                                │
│                                                                                                                 │
│  The EU AI Act entered into force on **1 August 2024**. Its application is staggered, meaning obligations take  │
│  effect in phases rather than all at once.                                                                      │
│                                                                                                                 │
│  ### **1. Already Applicable (Since 2 February 2025)**                                                          │
│  The first major milestone passed on February 2, 2025. As of this date, the following provisions are in force:  │
│  *   **Prohibited AI Practices:** AI systems that pose an unacceptable risk (e.g., social scoring, certain      │
│  forms of predictive policing, untargeted scraping of facial images) are now strictly banned.                   │
│  *   **General Provisions:** Definitions, governance structures, and the establishment of the AI Office.        │
│  *   **AI Literacy:** Obligations for providers and deployers to ensure staff dealing with the operation and    │
│  use of AI systems have a sufficient level of AI literacy.                                                      │
│  *   **Penalties:** Rules regarding enforcement and fines for non-compliance are now applicable.                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### **2. Upcoming Enforcement Deadlines**                                                                      │
│  The implementation continues over the next two years with several critical dates:                              │
│                                                                                                                 │
│  *   **2 August 2025 (12 months post-entry into force):**                                                       │
│      *   **GPAI (General Purpose AI) Rules:** Requirements for providers of General Purpose AI models           │
│  (including transparency, copyright adherence, and technical documentation) become applicable.                  │
│      *   **Governance:** Appointment of national competent authorities and the designation of the AI Office.    │
│                                                                                                                 │
│  *   **2 August 2026 (24 months post-entry into force):**                                                       │
│      *   **High-Risk AI Systems:** Requirements under Annex III (systems used in critical infrastructure,       │
│  education, employment, etc.) become applicable.                                                                │
│      *   **Regulatory Sandboxes:** Member States must have at least one AI regulatory sandbox operational at    │
│  the national level.                                                                                            │
│                                                        

As an EU AI Act researcher, I have summarized the enforcement timeline for you. 

The EU AI Act entered into force on **1 August 2024**. Its application is staggered, meaning obligations take effect in phases rather than all at once.

### **1. Already Applicable (Since 2 February 2025)**
The first major milestone passed on February 2, 2025. As of this date, the following provisions are in force:
*   **Prohibited AI Practices:** AI systems that pose an unacceptable risk (e.g., social scoring, certain forms of predictive policing, untargeted scraping of facial images) are now strictly banned.
*   **General Provisions:** Definitions, governance structures, and the establishment of the AI Office.
*   **AI Literacy:** Obligations for providers and deployers to ensure staff dealing with the operation and use of AI systems have a sufficient level of AI literacy.
*   **Penalties:** Rules regarding enforcement and fines for non-compliance are now applicable.

---

### **2. Upcoming Enforcement 

╭──────────────────────────────────────────── ✅ LiteAgent Completed ─────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Completed                                                                                            │
│  Role: EU AI Act Senior Data Researcher                                                                         │
│  id: 4cfae625-454e-4f10-9a0f-90894b70eb3d                                                                       │
│  role: EU AI Act Senior Data Researcher                                                                         │
│  goal: Uncover cutting-edge developments in EU AI Act                                                           │
│  backstory: You're a seasoned researcher with a knack for uncovering the latest developments in EU AI Act.      │
│  Known for your ability to find the most relevant information and present it in a clear and concise manner.     │
│  tools: [SerperDevTool(name='Search the internet with Serper', description='Tool Name:                          │
│  search_the_internet_with_serper\nTool Arguments: {\n  "description": "Input for SerperDevTool.",\n             │
│  "properties": {\n    "search_query": {\n      "description": "Mandatory search query you want to use to        │
│  search the internet",\n      "title": "Search Query",\n      "type": "string"\n    }\n  },\n  "required": [\n  │
│  "search_query"\n  ],\n  "title": "SerperDevToolSchema",\n  "type": "object",\n  "additionalProperties":        │
│  false\n}\nTool Description: A tool that can be used to search the internet with a search_query. Supports       │
│  different search types: \'search\' (default), \'news\'', env_vars=[EnvVar(name='SERPER_API_KEY',               │
│  description='API key for Serper', required=True, default=None)], args_schema=<class                            │
│  'crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevToolSchema'>, description_updated=False,          │
│  cache_function=<function BaseTool.<lambda> at 0x1132df240>, result_as_answer=False, max_usage_count=None,      │
│  current_usage_count=1, base_url='https://google.serper.dev', n_results=10, save_file=False,                    │
│  search_type='search', country='', location='', locale='')]                                                     │
│  verbose: True                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Your task

1. Run the cell. Check the verbose log: can you see when the agent calls the search tool, and what query it used? Does it search once, or multiple times with refined queries?

2. Compare this answer to Step 09's Researcher answering a similar question *without* a tool. Does Step 09 hedge, guess, or risk being stale on specific dates — while this one cites something concrete and current?

3. Now remove the tool (`tools=[]`) and rerun the exact same question. Does the agent admit it doesn't know the current deadlines, or guess anyway?

4. Swap in your own team's topic.

5. Note what you observed — it's evidence for `REPORT.md`'s Section 4.2 (Key Components: Tools).

## Shortcomings

`SerperDevTool` is one specific, built-in tool — wired up for you, calling one specific search API. Real-world tools are often bespoke (an internal database, a proprietary API, a legacy system) and don't ship with `crewai_tools`. There's also no standard way here to reuse a tool you wrote across different frameworks or projects.

[Step 11](step_11_mcp.ipynb) addresses that: connecting an agent to tools exposed by a separate process over a standard protocol, instead of a tool being built directly into the framework.

## Resources for further reading

- Schick, T., et al. (2023). *Toolformer: Language Models Can Teach Themselves to Use Tools*. [arXiv:2302.04761](https://arxiv.org/abs/2302.04761)
- [CrewAI Tools docs](https://docs.crewai.com/en/concepts/tools) — the full list of built-in tools beyond `SerperDevTool`, also summarized in the main [README](../../README.md#adding-more-tools-or-rag-for-students)